# 08_04 EES Framework – Actionability Score

## Purpose

This notebook calculates the **Actionability pillar** of the Explainability-Ethics Scoring (EES) framework.

It is configurable across German Credit, Pima Diabetes, and COMPAS by changing only `ACTIVE_DATASET`.

## Revised Actionability Definition

Actionability measures whether unfavourable model outcomes are practically contestable and actionable.

```text
Actionability Score = (Availability Score + Proximity Score + Sparsity Score) / 3
```

### Availability
Can a valid counterfactual be found?

### Proximity
How numerically close is the nearest counterfactual? This uses **minimum normalized numeric distance** from Notebook 07.

### Sparsity
How few features need to change? Fewer changed features indicate simpler and more understandable recourse.

Categorical variables are not used in proximity because category-to-category distance is subjective. Categorical changes are captured through sparsity.

In [1]:
# =========================================================
# 0. Dataset Configuration
# =========================================================

ACTIVE_DATASET = "german_credit"
# ACTIVE_DATASET = "pima_diabetes"
# ACTIVE_DATASET = "compas"

DATASET_CONFIGS = {
    "german_credit": {
        "dataset_name": "german_credit",
        "dataset_label": "German Credit",
        "target_name": "target",
        "positive_class_label": "Default",
        "negative_class_label": "Non-default",
        "results_folder": "german_credit",
        "counterfactual_folder": "07_counterfactual_fairness_dice_and_recourse_equity_german_credit",
        "ees_input_folder": "08_01_ees_framework_prepare_inputs_german_credit",
        "actionability_folder": "08_04_ees_framework_actionability_score_german_credit"
    },
    "pima_diabetes": {
        "dataset_name": "pima_diabetes",
        "dataset_label": "Pima Diabetes",
        "target_name": "target",
        "positive_class_label": "Diabetes",
        "negative_class_label": "No diabetes",
        "results_folder": "pima_diabetes",
        "counterfactual_folder": "07_counterfactual_fairness_dice_and_recourse_equity_pima_diabetes",
        "ees_input_folder": "08_01_ees_framework_prepare_inputs_pima_diabetes",
        "actionability_folder": "08_04_ees_framework_actionability_score_pima_diabetes"
    },
    "compas": {
        "dataset_name": "compas",
        "dataset_label": "COMPAS",
        "target_name": "target",
        "positive_class_label": "Recidivism",
        "negative_class_label": "No recidivism",
        "results_folder": "compas",
        "counterfactual_folder": "07_counterfactual_fairness_dice_and_recourse_equity_compas",
        "ees_input_folder": "08_01_ees_framework_prepare_inputs_compas",
        "actionability_folder": "08_04_ees_framework_actionability_score_compas"
    }
}

CONFIG = DATASET_CONFIGS[ACTIVE_DATASET]
print("Active dataset:", CONFIG["dataset_label"])
print("Dataset name:", CONFIG["dataset_name"])

Active dataset: German Credit
Dataset name: german_credit


In [2]:
# =========================================================
# 1. Import Required Libraries
# =========================================================

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
from datetime import datetime
import json
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 250)
pd.set_option("display.float_format", lambda x: f"{x:.6f}")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Libraries imported successfully.")
print("Random seed:", RANDOM_STATE)

Libraries imported successfully.
Random seed: 42


In [3]:
# =========================================================
# 2. Mount Google Drive and Configure Paths
# =========================================================

from google.colab import drive

try:
    drive.mount('/content/drive')
except Exception as e:
    print("Drive mount skipped or already available:", e)

ROOT_PATH = Path("/content/drive/MyDrive/Master_Research")

DATASET_NAME = CONFIG["dataset_name"]
NOTEBOOK_NAME = CONFIG["actionability_folder"]
RESULTS_ROOT = ROOT_PATH / "07_Results" / CONFIG["results_folder"]

COUNTERFACTUAL_RESULTS_PATH = RESULTS_ROOT / CONFIG["counterfactual_folder"]
COUNTERFACTUAL_TABLES_PATH = COUNTERFACTUAL_RESULTS_PATH / "tables"
COUNTERFACTUAL_DATASETS_PATH = COUNTERFACTUAL_RESULTS_PATH / "datasets"

EES_INPUT_RESULTS_PATH = RESULTS_ROOT / CONFIG["ees_input_folder"]
EES_INPUT_TABLES_PATH = EES_INPUT_RESULTS_PATH / "tables"
EES_INPUT_DATASETS_PATH = EES_INPUT_RESULTS_PATH / "datasets"

RESULTS_PATH = RESULTS_ROOT / NOTEBOOK_NAME
TABLES_PATH = RESULTS_PATH / "tables"
DATASETS_PATH = RESULTS_PATH / "datasets"
LOGS_PATH = RESULTS_PATH / "logs"

for path in [RESULTS_PATH, TABLES_PATH, DATASETS_PATH, LOGS_PATH]:
    path.mkdir(parents=True, exist_ok=True)

print("Results root:", RESULTS_ROOT)
print("Counterfactual path:", COUNTERFACTUAL_RESULTS_PATH)
print("Actionability output path:", RESULTS_PATH)

Mounted at /content/drive
Results root: /content/drive/MyDrive/Master_Research/07_Results/german_credit
Counterfactual path: /content/drive/MyDrive/Master_Research/07_Results/german_credit/07_counterfactual_fairness_dice_and_recourse_equity_german_credit
Actionability output path: /content/drive/MyDrive/Master_Research/07_Results/german_credit/08_04_ees_framework_actionability_score_german_credit


In [4]:
# =========================================================
# 3. Helper Functions
# =========================================================

def read_csv_safely(path, label, required=True):
    path = Path(path)
    if not path.exists():
        if required:
            raise FileNotFoundError(f"{label} not found at: {path}")
        print(f"Optional file not found: {label} at {path}")
        return pd.DataFrame()
    return pd.read_csv(path)


def find_first_existing_file(candidate_paths, label, required=True):
    for path in candidate_paths:
        path = Path(path)
        if path.exists():
            print(f"{label} found at: {path}")
            return path
    if required:
        checked = "\n".join(str(path) for path in candidate_paths)
        raise FileNotFoundError(f"{label} not found. Checked:\n{checked}")
    print(f"{label} not found.")
    return None


def save_df(df, filename, folder=None, index=False):
    if folder is None:
        folder = TABLES_PATH
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    output_path = folder / filename
    df.to_csv(output_path, index=index)
    print(f"Saved: {output_path}")
    return output_path


def save_excel(dfs, filename, folder=None):
    if folder is None:
        folder = TABLES_PATH
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    output_path = folder / filename
    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        for sheet_name, df in dfs.items():
            if isinstance(df, pd.DataFrame):
                df.to_excel(writer, sheet_name=sheet_name[:31], index=False)
    print(f"Saved: {output_path}")
    return output_path


def classify_actionability(score):
    if pd.isna(score):
        return "Not available"
    if score >= 0.80:
        return "High actionability"
    if score >= 0.60:
        return "Moderate-high actionability"
    if score >= 0.40:
        return "Moderate actionability"
    if score >= 0.20:
        return "Low-moderate actionability"
    return "Low actionability"


def clip_01(series):
    return pd.Series(series).astype(float).clip(0, 1)

## 4. Load Counterfactual Outputs

This notebook consumes outputs from Notebook 07. If Notebook 07 already produced the revised three-pillar score, this notebook validates and standardises it. If not, it recalculates the score from instance-level evidence.

In [5]:
# =========================================================
# 4. Load Counterfactual Outputs
# =========================================================

actionability_score_file = find_first_existing_file(
    [
        COUNTERFACTUAL_DATASETS_PATH / "ees_actionability_model_scores.csv",
        COUNTERFACTUAL_TABLES_PATH / "ees_actionability_model_scores.csv",
        COUNTERFACTUAL_DATASETS_PATH / "actionability_score_summary.csv",
        COUNTERFACTUAL_TABLES_PATH / "actionability_score_summary.csv",
        EES_INPUT_DATASETS_PATH / "ees_actionability_input.csv",
        EES_INPUT_TABLES_PATH / "ees_actionability_input.csv"
    ],
    "Actionability model score output",
    required=False
)

counterfactual_instance_file = find_first_existing_file(
    [
        COUNTERFACTUAL_DATASETS_PATH / "counterfactual_instance_summary.csv",
        COUNTERFACTUAL_TABLES_PATH / "counterfactual_instance_summary.csv"
    ],
    "Counterfactual instance summary",
    required=False
)

counterfactual_long_file = find_first_existing_file(
    [
        COUNTERFACTUAL_DATASETS_PATH / "counterfactual_long_format.csv",
        COUNTERFACTUAL_TABLES_PATH / "counterfactual_long_format.csv"
    ],
    "Counterfactual long format",
    required=False
)

actionability_model_scores_raw_df = read_csv_safely(actionability_score_file, "Actionability model score output", required=False) if actionability_score_file else pd.DataFrame()
counterfactual_instance_summary_df = read_csv_safely(counterfactual_instance_file, "Counterfactual instance summary", required=False) if counterfactual_instance_file else pd.DataFrame()
counterfactual_long_df = read_csv_safely(counterfactual_long_file, "Counterfactual long format", required=False) if counterfactual_long_file else pd.DataFrame()

print("actionability_model_scores_raw_df:", actionability_model_scores_raw_df.shape)
print("counterfactual_instance_summary_df:", counterfactual_instance_summary_df.shape)
print("counterfactual_long_df:", counterfactual_long_df.shape)

display(actionability_model_scores_raw_df.head())
display(counterfactual_instance_summary_df.head())

Actionability model score output found at: /content/drive/MyDrive/Master_Research/07_Results/german_credit/07_counterfactual_fairness_dice_and_recourse_equity_german_credit/datasets/ees_actionability_model_scores.csv
Counterfactual instance summary found at: /content/drive/MyDrive/Master_Research/07_Results/german_credit/07_counterfactual_fairness_dice_and_recourse_equity_german_credit/datasets/counterfactual_instance_summary.csv
Counterfactual long format found at: /content/drive/MyDrive/Master_Research/07_Results/german_credit/07_counterfactual_fairness_dice_and_recourse_equity_german_credit/datasets/counterfactual_long_format.csv
actionability_model_scores_raw_df: (5, 16)
counterfactual_instance_summary_df: (251, 10)
counterfactual_long_df: (753, 25)


,model_id,model_name,n_unfavourable_instances,n_skipped_due_to_missing_values,n_actionable_recourse_instances,mean_min_normalized_numeric_distance,median_min_normalized_numeric_distance,mean_changed_features,median_changed_features,availability_score,proximity_score,sparsity_score,actionability_score,actionability_score_pct,actionability_rank,actionability_band
0,decision_tree,Decision Tree,68,0,68,0.025291,0.007344,1.382353,1.000000,1.000000,0.974709,0.846405,0.940371,94.037147,1,High actionability
1,xgboost,XGBoost,49,0,49,0.044535,0.035714,1.265306,1.000000,1.000000,0.955465,0.859410,0.938292,93.829177,2,High actionability
2,random_forest,Random Forest,39,0,39,0.036329,0.023438,1.512821,2.000000,1.000000,0.963671,0.831909,0.931860,93.185992,3,High actionability
3,mlp,MLP,56,0,56,0.053485,0.044850,1.375000,1.000000,1.000000,0.946515,0.847222,0.931246,93.124584,4,High actionability
4,logistic_regression,Logistic Regression,39,0,39,0.046948,0.025879,1.641026,2.000000,1.000000,0.953052,0.817664,0.923572,92.357181,5,High actionability


,model_id,model_name,test_instance_position,counterfactual_found,skipped_due_to_missing_values,n_counterfactuals,min_normalized_numeric_distance,mean_normalized_numeric_distance,min_changed_features,mean_changed_features
0,logistic_regression,Logistic Regression,14,True,False,3,0.000000,0.088648,2.000000,2.000000
1,logistic_regression,Logistic Regression,23,True,False,3,0.000000,0.045635,2.000000,2.000000
2,logistic_regression,Logistic Regression,25,True,False,3,0.000000,0.060087,2.000000,2.000000
3,logistic_regression,Logistic Regression,33,True,False,3,0.147321,0.202439,2.000000,2.000000
4,logistic_regression,Logistic Regression,37,True,False,3,0.160714,0.168155,2.000000,2.000000


In [6]:
# =========================================================
# 5. Validate Required Inputs
# =========================================================

validation_rows = []
validation_rows.append({
    "check": "Either model-level actionability or instance-level counterfactual summary exists",
    "passed": (not actionability_model_scores_raw_df.empty) or (not counterfactual_instance_summary_df.empty)
})

if not actionability_model_scores_raw_df.empty:
    for col in ["model_id", "model_name"]:
        validation_rows.append({"check": f"model-level actionability has {col}", "passed": col in actionability_model_scores_raw_df.columns})

if not counterfactual_instance_summary_df.empty:
    required_instance_cols = ["model_id", "model_name", "test_instance_position", "counterfactual_found", "min_normalized_numeric_distance", "min_changed_features"]
    for col in required_instance_cols:
        validation_rows.append({"check": f"counterfactual instance summary has {col}", "passed": col in counterfactual_instance_summary_df.columns})

actionability_input_validation_df = pd.DataFrame(validation_rows)
display(actionability_input_validation_df)
save_df(actionability_input_validation_df, "actionability_input_validation.csv")

if not actionability_input_validation_df["passed"].all():
    raise ValueError("One or more actionability input validation checks failed.")

,check,passed
0,Either model-level actionability or instance-l...,True
1,model-level actionability has model_id,True
2,model-level actionability has model_name,True
3,counterfactual instance summary has model_id,True
4,counterfactual instance summary has model_name,True
5,counterfactual instance summary has test_insta...,True
6,counterfactual instance summary has counterfac...,True
7,counterfactual instance summary has min_normal...,True
8,counterfactual instance summary has min_change...,True


Saved: /content/drive/MyDrive/Master_Research/07_Results/german_credit/08_04_ees_framework_actionability_score_german_credit/tables/actionability_input_validation.csv


## 6. Recalculate Three-Pillar Actionability Score

The score is recalculated from instance-level counterfactual evidence where available. This ensures that Notebook 08_04 is aligned to the revised Actionability definition even if older Notebook 07 outputs are present.

In [7]:
# =========================================================
# 6. Recalculate Three-Pillar Actionability Score
# =========================================================

if counterfactual_instance_summary_df.empty:
    recalculated_actionability_df = pd.DataFrame()
else:
    temp_instance_df = counterfactual_instance_summary_df.copy()
    temp_instance_df["counterfactual_found"] = temp_instance_df["counterfactual_found"].astype(bool)

    recalculated_actionability_df = (
        temp_instance_df
        .groupby(["model_id", "model_name"], as_index=False)
        .agg(
            n_unfavourable_instances=("test_instance_position", "nunique"),
            n_actionable_recourse_instances=("counterfactual_found", "sum"),
            mean_min_normalized_numeric_distance=("min_normalized_numeric_distance", "mean"),
            median_min_normalized_numeric_distance=("min_normalized_numeric_distance", "median"),
            mean_changed_features=("min_changed_features", "mean"),
            median_changed_features=("min_changed_features", "median")
        )
    )

    if "skipped_due_to_missing_values" in temp_instance_df.columns:
        skipped_df = (
            temp_instance_df
            .groupby(["model_id", "model_name"], as_index=False)
            .agg(n_skipped_due_to_missing_values=("skipped_due_to_missing_values", "sum"))
        )
        recalculated_actionability_df = recalculated_actionability_df.merge(skipped_df, on=["model_id", "model_name"], how="left")
    else:
        recalculated_actionability_df["n_skipped_due_to_missing_values"] = 0

    recalculated_actionability_df["availability_score"] = (
        recalculated_actionability_df["n_actionable_recourse_instances"] /
        recalculated_actionability_df["n_unfavourable_instances"]
    )

    recalculated_actionability_df["proximity_score"] = clip_01(
        1 - recalculated_actionability_df["mean_min_normalized_numeric_distance"]
    )

    if not counterfactual_long_df.empty:
        original_feature_cols = [col for col in counterfactual_long_df.columns if col.startswith("original_")]
        total_features = len(original_feature_cols)
    else:
        total_features = np.nan

    if pd.isna(total_features) or total_features == 0:
        total_features = 1

    recalculated_actionability_df["total_features_for_sparsity"] = total_features

    recalculated_actionability_df["sparsity_score"] = clip_01(
        1 - (recalculated_actionability_df["mean_changed_features"] / recalculated_actionability_df["total_features_for_sparsity"])
    )

    recalculated_actionability_df["actionability_score"] = (
        recalculated_actionability_df["availability_score"] +
        recalculated_actionability_df["proximity_score"] +
        recalculated_actionability_df["sparsity_score"]
    ) / 3

    recalculated_actionability_df["actionability_score_pct"] = recalculated_actionability_df["actionability_score"] * 100

    recalculated_actionability_df["actionability_rank"] = (
        recalculated_actionability_df["actionability_score"]
        .rank(ascending=False, method="dense")
        .astype(int)
    )

    recalculated_actionability_df["actionability_band"] = recalculated_actionability_df["actionability_score"].apply(classify_actionability)

    recalculated_actionability_df = (
        recalculated_actionability_df
        .sort_values(["actionability_rank", "model_id"])
        .reset_index(drop=True)
    )

display(recalculated_actionability_df)

,model_id,model_name,n_unfavourable_instances,n_actionable_recourse_instances,mean_min_normalized_numeric_distance,median_min_normalized_numeric_distance,mean_changed_features,median_changed_features,n_skipped_due_to_missing_values,availability_score,proximity_score,total_features_for_sparsity,sparsity_score,actionability_score,actionability_score_pct,actionability_rank,actionability_band
0,decision_tree,Decision Tree,68,68,0.025291,0.007344,1.382353,1.000000,0,1.000000,0.974709,9,0.846405,0.940371,94.037147,1,High actionability
1,xgboost,XGBoost,49,49,0.044535,0.035714,1.265306,1.000000,0,1.000000,0.955465,9,0.859410,0.938292,93.829177,2,High actionability
2,random_forest,Random Forest,39,39,0.036329,0.023438,1.512821,2.000000,0,1.000000,0.963671,9,0.831909,0.931860,93.185992,3,High actionability
3,mlp,MLP,56,56,0.053485,0.044850,1.375000,1.000000,0,1.000000,0.946515,9,0.847222,0.931246,93.124584,4,High actionability
4,logistic_regression,Logistic Regression,39,39,0.046948,0.025879,1.641026,2.000000,0,1.000000,0.953052,9,0.817664,0.923572,92.357181,5,High actionability


In [8]:
# =========================================================
# 7. Select Final Actionability Scores
# =========================================================

if not recalculated_actionability_df.empty:
    final_actionability_score_df = recalculated_actionability_df.copy()
    final_actionability_score_df["score_source"] = "recalculated_from_instance_level_counterfactuals"
else:
    final_actionability_score_df = actionability_model_scores_raw_df.copy()
    final_actionability_score_df["score_source"] = "loaded_from_notebook_07"

required_final_cols = [
    "availability_score", "proximity_score", "sparsity_score",
    "actionability_score", "actionability_score_pct",
    "actionability_rank", "actionability_band"
]

for col in required_final_cols:
    if col not in final_actionability_score_df.columns:
        final_actionability_score_df[col] = np.nan

final_actionability_score_df["dataset_name"] = DATASET_NAME
final_actionability_score_df["dataset_label"] = CONFIG["dataset_label"]

front_cols = [
    "dataset_name", "dataset_label", "model_id", "model_name",
    "availability_score", "proximity_score", "sparsity_score",
    "actionability_score", "actionability_score_pct",
    "actionability_rank", "actionability_band", "score_source"
]
remaining_cols = [col for col in final_actionability_score_df.columns if col not in front_cols]
final_actionability_score_df = final_actionability_score_df[front_cols + remaining_cols]

display(final_actionability_score_df)

,dataset_name,dataset_label,model_id,model_name,availability_score,proximity_score,sparsity_score,actionability_score,actionability_score_pct,actionability_rank,actionability_band,score_source,n_unfavourable_instances,n_actionable_recourse_instances,mean_min_normalized_numeric_distance,median_min_normalized_numeric_distance,mean_changed_features,median_changed_features,n_skipped_due_to_missing_values,total_features_for_sparsity
0,german_credit,German Credit,decision_tree,Decision Tree,1.000000,0.974709,0.846405,0.940371,94.037147,1,High actionability,recalculated_from_instance_level_counterfactuals,68,68,0.025291,0.007344,1.382353,1.000000,0,9
1,german_credit,German Credit,xgboost,XGBoost,1.000000,0.955465,0.859410,0.938292,93.829177,2,High actionability,recalculated_from_instance_level_counterfactuals,49,49,0.044535,0.035714,1.265306,1.000000,0,9
2,german_credit,German Credit,random_forest,Random Forest,1.000000,0.963671,0.831909,0.931860,93.185992,3,High actionability,recalculated_from_instance_level_counterfactuals,39,39,0.036329,0.023438,1.512821,2.000000,0,9
3,german_credit,German Credit,mlp,MLP,1.000000,0.946515,0.847222,0.931246,93.124584,4,High actionability,recalculated_from_instance_level_counterfactuals,56,56,0.053485,0.044850,1.375000,1.000000,0,9
4,german_credit,German Credit,logistic_regression,Logistic Regression,1.000000,0.953052,0.817664,0.923572,92.357181,5,High actionability,recalculated_from_instance_level_counterfactuals,39,39,0.046948,0.025879,1.641026,2.000000,0,9


In [9]:
# =========================================================
# 8. Validate Against Notebook 07 Model-Level Score
# =========================================================

if recalculated_actionability_df.empty or actionability_model_scores_raw_df.empty:
    actionability_score_reconciliation_df = pd.DataFrame()
else:
    loaded_cols = [col for col in [
        "model_id", "model_name", "availability_score", "proximity_score", "sparsity_score", "actionability_score"
    ] if col in actionability_model_scores_raw_df.columns]

    loaded_compare_df = actionability_model_scores_raw_df[loaded_cols].copy()
    recalculated_compare_df = recalculated_actionability_df[[
        "model_id", "availability_score", "proximity_score", "sparsity_score", "actionability_score"
    ]].copy()

    actionability_score_reconciliation_df = loaded_compare_df.merge(
        recalculated_compare_df,
        on="model_id",
        how="inner",
        suffixes=("_loaded", "_recalculated")
    )

    for metric in ["availability_score", "proximity_score", "sparsity_score", "actionability_score"]:
        loaded_col = f"{metric}_loaded"
        recalculated_col = f"{metric}_recalculated"
        if loaded_col in actionability_score_reconciliation_df.columns and recalculated_col in actionability_score_reconciliation_df.columns:
            actionability_score_reconciliation_df[f"{metric}_difference"] = (
                actionability_score_reconciliation_df[loaded_col] -
                actionability_score_reconciliation_df[recalculated_col]
            ).abs()

display(actionability_score_reconciliation_df)

,model_id,model_name,availability_score_loaded,proximity_score_loaded,sparsity_score_loaded,actionability_score_loaded,availability_score_recalculated,proximity_score_recalculated,sparsity_score_recalculated,actionability_score_recalculated,availability_score_difference,proximity_score_difference,sparsity_score_difference,actionability_score_difference
0,decision_tree,Decision Tree,1.000000,0.974709,0.846405,0.940371,1.000000,0.974709,0.846405,0.940371,0.000000,0.000000,0.000000,0.000000
1,xgboost,XGBoost,1.000000,0.955465,0.859410,0.938292,1.000000,0.955465,0.859410,0.938292,0.000000,0.000000,0.000000,0.000000
2,random_forest,Random Forest,1.000000,0.963671,0.831909,0.931860,1.000000,0.963671,0.831909,0.931860,0.000000,0.000000,0.000000,0.000000
3,mlp,MLP,1.000000,0.946515,0.847222,0.931246,1.000000,0.946515,0.847222,0.931246,0.000000,0.000000,0.000000,0.000000
4,logistic_regression,Logistic Regression,1.000000,0.953052,0.817664,0.923572,1.000000,0.953052,0.817664,0.923572,0.000000,0.000000,0.000000,0.000000


## 9. Component Interpretation Table

In [10]:
# =========================================================
# 9. Component Interpretation Table
# =========================================================

actionability_component_interpretation_df = pd.DataFrame([
    {
        "component": "Availability Score",
        "question_answered": "Can recourse be found?",
        "calculation": "n_actionable_recourse_instances / n_unfavourable_instances",
        "higher_is_better": True,
        "interpretation": "Higher values mean more unfavourable cases have at least one valid counterfactual."
    },
    {
        "component": "Proximity Score",
        "question_answered": "How close is the nearest counterfactual?",
        "calculation": "1 - mean_min_normalized_numeric_distance",
        "higher_is_better": True,
        "interpretation": "Higher values mean counterfactuals require smaller normalized numeric changes."
    },
    {
        "component": "Sparsity Score",
        "question_answered": "How few features need to change?",
        "calculation": "1 - (mean_changed_features / total_features)",
        "higher_is_better": True,
        "interpretation": "Higher values mean fewer attributes must change."
    },
    {
        "component": "Final Actionability Score",
        "question_answered": "Is recourse available, close and simple?",
        "calculation": "(availability_score + proximity_score + sparsity_score) / 3",
        "higher_is_better": True,
        "interpretation": "Equal-weight composite score summarising practical actionability."
    }
])

display(actionability_component_interpretation_df)

,component,question_answered,calculation,higher_is_better,interpretation
0,Availability Score,Can recourse be found?,n_actionable_recourse_instances / n_unfavourab...,True,Higher values mean more unfavourable cases hav...
1,Proximity Score,How close is the nearest counterfactual?,1 - mean_min_normalized_numeric_distance,True,Higher values mean counterfactuals require sma...
2,Sparsity Score,How few features need to change?,1 - (mean_changed_features / total_features),True,Higher values mean fewer attributes must change.
3,Final Actionability Score,"Is recourse available, close and simple?",(availability_score + proximity_score + sparsi...,True,Equal-weight composite score summarising pract...


In [11]:
# =========================================================
# 10. Save Actionability Outputs
# =========================================================

save_df(final_actionability_score_df, "ees_actionability_model_scores.csv")
save_df(final_actionability_score_df, "actionability_score_summary.csv")
save_df(recalculated_actionability_df, "recalculated_actionability_score_summary.csv")
save_df(actionability_score_reconciliation_df, "actionability_score_reconciliation.csv")
save_df(actionability_component_interpretation_df, "actionability_component_interpretation.csv")

save_df(final_actionability_score_df, "ees_actionability_model_scores.csv", folder=DATASETS_PATH)
save_df(final_actionability_score_df, "actionability_score_summary.csv", folder=DATASETS_PATH)
save_df(recalculated_actionability_df, "recalculated_actionability_score_summary.csv", folder=DATASETS_PATH)
save_df(actionability_score_reconciliation_df, "actionability_score_reconciliation.csv", folder=DATASETS_PATH)
save_df(actionability_component_interpretation_df, "actionability_component_interpretation.csv", folder=DATASETS_PATH)

save_excel(
    {
        "final_actionability": final_actionability_score_df,
        "recalculated_actionability": recalculated_actionability_df,
        "reconciliation": actionability_score_reconciliation_df,
        "component_interpretation": actionability_component_interpretation_df,
        "input_validation": actionability_input_validation_df,
        "raw_model_scores": actionability_model_scores_raw_df,
        "instance_summary": counterfactual_instance_summary_df
    },
    f"ees_actionability_score_{DATASET_NAME}_outputs.xlsx"
)

manifest = {
    "dataset_name": DATASET_NAME,
    "dataset_label": CONFIG["dataset_label"],
    "notebook_name": NOTEBOOK_NAME,
    "run_timestamp": datetime.now().isoformat(),
    "random_state": RANDOM_STATE,
    "actionability_formula": "(availability_score + proximity_score + sparsity_score) / 3",
    "availability_definition": "n_actionable_recourse_instances / n_unfavourable_instances",
    "proximity_definition": "1 - mean minimum normalized numeric distance",
    "sparsity_definition": "1 - mean changed feature count divided by total number of features",
    "standardisation_note": "This actionability notebook uses the same configurable three-pillar actionability formula across German Credit, Pima Diabetes and COMPAS."
}

with open(LOGS_PATH / "ees_actionability_score_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=4)

with open(TABLES_PATH / "ees_actionability_score_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=4)

print("All actionability outputs saved.")

Saved: /content/drive/MyDrive/Master_Research/07_Results/german_credit/08_04_ees_framework_actionability_score_german_credit/tables/ees_actionability_model_scores.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/german_credit/08_04_ees_framework_actionability_score_german_credit/tables/actionability_score_summary.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/german_credit/08_04_ees_framework_actionability_score_german_credit/tables/recalculated_actionability_score_summary.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/german_credit/08_04_ees_framework_actionability_score_german_credit/tables/actionability_score_reconciliation.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/german_credit/08_04_ees_framework_actionability_score_german_credit/tables/actionability_component_interpretation.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/german_credit/08_04_ees_framework_actionability_score_german_credit/datasets/ees_actionab

## 11. Interpretation Note

The revised Actionability pillar evaluates whether unfavourable model outcomes are practically contestable.

- High availability means recourse exists for many unfavourable cases.
- High proximity means counterfactuals are numerically close.
- High sparsity means counterfactuals require few changes.

This score does not imply that counterfactuals are automatically feasible in the real world. It provides model-based evidence of contestability and recourse quality.

In [12]:
# =========================================================
# 11. Final Completion Message
# =========================================================

print("=" * 100)
print(f"08_04 Actionability Score completed successfully for: {CONFIG['dataset_label']}")
print("=" * 100)

print(f"Tables saved to:   {TABLES_PATH}")
print(f"Datasets saved to: {DATASETS_PATH}")
print(f"Logs saved to:     {LOGS_PATH}")

print("\nKey outputs:")
print("- ees_actionability_model_scores.csv")
print("- actionability_score_summary.csv")
print("- recalculated_actionability_score_summary.csv")
print("- actionability_score_reconciliation.csv")
print(f"- ees_actionability_score_{DATASET_NAME}_outputs.xlsx")

08_04 Actionability Score completed successfully for: German Credit
Tables saved to:   /content/drive/MyDrive/Master_Research/07_Results/german_credit/08_04_ees_framework_actionability_score_german_credit/tables
Datasets saved to: /content/drive/MyDrive/Master_Research/07_Results/german_credit/08_04_ees_framework_actionability_score_german_credit/datasets
Logs saved to:     /content/drive/MyDrive/Master_Research/07_Results/german_credit/08_04_ees_framework_actionability_score_german_credit/logs

Key outputs:
- ees_actionability_model_scores.csv
- actionability_score_summary.csv
- recalculated_actionability_score_summary.csv
- actionability_score_reconciliation.csv
- ees_actionability_score_german_credit_outputs.xlsx
